In [ ]:
"""
====================================================================
Estudos_Queimadas — Pipeline de consolidacao e analise
====================================================================
Consolida dados meteorologicos (INMET) e focos de calor (INPE/
BDQueimadas) para os 645 municipios do estado de Sao Paulo
(2015-2024), e avalia quais variaveis meteorologicas sao
estatisticamente significantes para a ocorrencia de focos de calor,
via regressao logistica.

Requisitos: pandas, numpy, requests, scikit-learn, statsmodels, scipy
    pip install pandas numpy requests scikit-learn statsmodels scipy

Saida: dataframe_final_estacao_hora.csv (na pasta do projeto) e o
resultado impresso da regressao logistica de significancia.
====================================================================
"""

import os
import requests
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.api as sm
from scipy import stats
from sklearn.neighbors import BallTree
from statsmodels.stats.outliers_influence import variance_inflation_factor

# --------------------------------------------------------------
# Configuracao -- ajuste os caminhos conforme sua estrutura de pastas
# --------------------------------------------------------------
PASTA_PROJETO = r"C:\Users\CBI2\Downloads\Estudos_Queimadas"
PASTA_INMET = os.path.join(PASTA_PROJETO, "INMET")
PASTA_FOCOS = os.path.join(PASTA_PROJETO, "Focos")

ANO_INICIO, ANO_FIM = 2015, 2024


# ================================================================
# ETAPA 1 -- Catalogo nacional de estacoes automaticas (INMET)
# ================================================================
# Le do arquivo local ja salvo (catalogo_estacoes_completo.csv, obtido
# anteriormente da API do INMET) em vez de baixar a cada execucao --
# evita depender de acesso a internet/rede que pode estar bloqueado
# (ex.: rede corporativa bloqueando apitempo.inmet.gov.br).
print("[Etapa 1] Lendo catalogo de estacoes do arquivo local...")
CAMINHO_CATALOGO_ESTACOES = os.path.join(PASTA_INMET, 'catalogo_estacoes_completo.csv')
estacoes = pd.read_csv(CAMINHO_CATALOGO_ESTACOES, encoding='utf-8-sig')

RENOMEIA_ESTACOES = {
    'DC_NOME': 'Nome', 'SG_ESTADO': 'UF', 'CD_SITUACAO': 'Situação',
    'VL_LATITUDE': 'Latitude', 'VL_LONGITUDE': 'Longitude',
    'VL_ALTITUDE': 'Altitude', 'DT_INICIO_OPERACAO': 'Data de Instalação',
    'CD_ESTACAO': 'Código',
}
# so renomeia as colunas que ainda estiverem com o nome original da API
# (caso o arquivo salvo ja tenha sido renomeado antes, nao ha problema)
estacoes = estacoes.rename(columns={k: v for k, v in RENOMEIA_ESTACOES.items() if k in estacoes.columns})


# ================================================================
# ETAPA 2 -- Municipios de SP e a estacao mais proxima de cada um
# ================================================================
# Coordenadas dos municipios: compilacao publica baseada em dados do
# IBGE (kelvins/municipios-brasileiros). codigo_uf 35 = Sao Paulo.
# Le de um arquivo local em cache se ja existir (mesma logica da
# Etapa 1, para nao depender de internet toda vez); se ainda nao
# existir, baixa uma vez e salva para as proximas execucoes.
print("[Etapa 2] Calculando estacao mais proxima de cada municipio...")
CAMINHO_MUNICIPIOS = os.path.join(PASTA_PROJETO, 'municipios_brasileiros.csv')
if os.path.exists(CAMINHO_MUNICIPIOS):
    municipios = pd.read_csv(CAMINHO_MUNICIPIOS)
else:
    municipios = pd.read_csv(
        "https://raw.githubusercontent.com/kelvins/municipios-brasileiros/main/csv/municipios.csv"
    )
    municipios.to_csv(CAMINHO_MUNICIPIOS, index=False)
municipios_sp = municipios[municipios['codigo_uf'] == 35].reset_index(drop=True)
assert len(municipios_sp) == 645, "Esperado 645 municipios de SP"

# Estacao mais proxima de cada municipio, por distancia haversine
# (BallTree). Usamos o catalogo NACIONAL como candidato, nao so
# estacoes "de SP", porque municipios de fronteira podem ter a
# estacao mais proxima em outro estado.
estacoes_rad = np.radians(estacoes[['Latitude', 'Longitude']].astype(float).values)
tree_nacional = BallTree(estacoes_rad, metric='haversine')
municipios_rad = np.radians(municipios_sp[['latitude', 'longitude']].values)
_, idx = tree_nacional.query(municipios_rad, k=1)
municipios_sp['Codigo_estacao_mais_proxima'] = estacoes.iloc[idx.flatten()]['Código'].values

# Conjunto de estacoes realmente relevantes: so as que sao "a mais
# proxima" de pelo menos 1 dos 645 municipios (nao todas que existem
# fisicamente na regiao) -- normalmente bem menos que 645, porque
# municipios vizinhos costumam compartilhar a mesma estacao.
codigos_estacoes = municipios_sp['Codigo_estacao_mais_proxima'].unique().tolist()
print(f"  -> {len(codigos_estacoes)} estacoes cobrem os 645 municipios de SP")


# ================================================================
# ETAPA 3 -- Consolidacao dos dados meteorologicos brutos (INMET)
# ================================================================
# Estrutura esperada: PASTA_INMET/<ano>/INMET_..._<codigo>_...CSV
# Uma pasta por ano (2015-2024), um arquivo por estacao dentro dela.
#
# Detalhes do formato bruto do INMET, descobertos na validacao:
# - Os dados comecam na linha 10 (9 linhas de metadado/cabecalho
#   antes disso).
# - Cada linha termina com ";" sobrando, criando um campo vazio
#   extra (por isso a lista de colunas tem 1 nome a mais, "_extra",
#   descartado logo depois -- sem isso, o pandas desloca todas as
#   colunas nomeadas 1 posicao para a esquerda do valor real).
# - O separador decimal e virgula, e o separador de campo e ";".
# - -9999 e o valor sentinela de "sem medicao" nas colunas numericas.
# - O formato de DATA/HORA muda entre alguns anos (ex.: "2016-01-01"/
#   "00:00" vs "2019/01/01"/"0000 UTC") -- por isso normalizamos os
#   dois formatos antes de converter para timestamp.
print("[Etapa 3] Consolidando dados meteorologicos brutos (pode levar alguns minutos)...")

COLUNAS_MET = [
    'DATA (YYYY-MM-DD)', 'HORA (UTC)', 'PRECIPITAÇÃO TOTAL, HORÁRIO (mm)',
    'PRESSAO ATMOSFERICA AO NIVEL DA ESTACAO, HORARIA (mB)',
    'PRESSÃO ATMOSFERICA MAX.NA HORA ANT. (AUT) (mB)',
    'PRESSÃO ATMOSFERICA MIN. NA HORA ANT. (AUT) (mB)', 'RADIACAO GLOBAL (KJ/m²)',
    'TEMPERATURA DO AR - BULBO SECO, HORARIA (°C)', 'TEMPERATURA DO PONTO DE ORVALHO (°C)',
    'TEMPERATURA MÁXIMA NA HORA ANT. (AUT) (°C)', 'TEMPERATURA MÍNIMA NA HORA ANT. (AUT) (°C)',
    'TEMPERATURA ORVALHO MAX. NA HORA ANT. (AUT) (°C)', 'TEMPERATURA ORVALHO MIN. NA HORA ANT. (AUT) (°C)',
    'UMIDADE REL. MAX. NA HORA ANT. (AUT) (%)', 'UMIDADE REL. MIN. NA HORA ANT. (AUT) (%)',
    'UMIDADE RELATIVA DO AR, HORARIA (%)', 'VENTO, DIREÇÃO HORARIA (gr) (° (gr))',
    'VENTO, RAJADA MAXIMA (m/s)', 'VENTO, VELOCIDADE HORARIA (m/s)',
]
COLUNAS_ARQUIVO_MET = COLUNAS_MET + ['_extra']

partes_met = []
for ano in range(ANO_INICIO, ANO_FIM + 1):
    pasta_ano = os.path.join(PASTA_INMET, str(ano))
    # os.listdir + filtro manual em vez de glob('*.CSV')+glob('*.csv'):
    # no Windows essas duas buscas dao o MESMO resultado (o sistema de
    # arquivos nao diferencia maiuscula/minuscula), o que duplicaria
    # cada arquivo se as duas fossem usadas juntas.
    arquivos = [os.path.join(pasta_ano, f) for f in os.listdir(pasta_ano) if f.lower().endswith('.csv')]
    for caminho in arquivos:
        codigo = next((c for c in codigos_estacoes if c in os.path.basename(caminho)), None)
        if codigo is None:
            continue  # nao e uma das estacoes relevantes para SP
        df = pd.read_csv(
            caminho, sep=';', skiprows=9, names=COLUNAS_ARQUIVO_MET,
            encoding='latin1', decimal=','
        ).drop(columns=['_extra'])
        df['Codigo_Estacao'] = codigo
        partes_met.append(df)

met = pd.concat(partes_met, ignore_index=True)

# timestamp unico, cobrindo os dois formatos de DATA/HORA observados
data_norm = met['DATA (YYYY-MM-DD)'].str.replace('/', '-', regex=False)
hora_limpa = (
    met['HORA (UTC)'].str.replace(' UTC', '', regex=False)
    .str.replace(':', '', regex=False).str.zfill(4)
)
met['timestamp'] = pd.to_datetime(
    data_norm + ' ' + hora_limpa.str[:2] + ':' + hora_limpa.str[2:], format='%Y-%m-%d %H:%M'
)
met['Codigo_Estacao'] = met['Codigo_Estacao'].str.strip().str.upper()

COLUNAS_NUMERICAS = [c for c in COLUNAS_MET if c not in ('DATA (YYYY-MM-DD)', 'HORA (UTC)')]
met[COLUNAS_NUMERICAS] = met[COLUNAS_NUMERICAS].replace(-9999, np.nan)

print(f"  -> {len(met)} linhas meteorologicas | {met['Codigo_Estacao'].nunique()} estacoes")


# ================================================================
# ETAPA 4 -- Preenchimento de lacunas curtas (falha de sensor)
# ================================================================
# Falhas de estacao inteira por algumas horas sao comuns na rede do
# INMET (reconhecido pelo proprio orgao, agravado durante a pandemia
# por causa de manutencoes adiadas). Preenchemos so lacunas de ate
# 6h, usando a leitura valida mais proxima da mesma estacao -- acima
# disso, a aproximacao deixa de ser razoavel e a ausencia e mantida
# como esta (NaN).
print("[Etapa 4] Preenchendo lacunas curtas (ate 6h) e tratando radiacao noturna...")
met = met.sort_values(['Codigo_Estacao', 'timestamp']).reset_index(drop=True)
met[COLUNAS_NUMERICAS] = (
    met.groupby('Codigo_Estacao')[COLUNAS_NUMERICAS]
    .transform(lambda s: s.ffill(limit=6).bfill(limit=6))
)

# Radiacao solar: boa parte do "sem medicao" durante a madrugada/
# noite (UTC 22h-8h ~ 19h-05h horario local) nao e falha de sensor,
# e ausencia real de sol -- zeramos essas horas especificamente,
# mantendo NaN so para falhas reais durante o dia.
hora_utc = met['timestamp'].dt.hour
noite = (hora_utc >= 22) | (hora_utc <= 8)
met.loc[noite & met['RADIACAO GLOBAL (KJ/m²)'].isna(), 'RADIACAO GLOBAL (KJ/m²)'] = 0


# ================================================================
# ETAPA 5 -- Consolidacao dos focos de calor (INPE/BDQueimadas)
# ================================================================
# Estrutura esperada:
#   PASTA_FOCOS/focos_br_sp_ref_<ano>/focos_br_sp_ref_<ano>.csv
print("[Etapa 5] Consolidando focos de calor...")
COLUNAS_FOCOS = ['id_bdq', 'foco_id', 'lat', 'lon', 'data_pas', 'pais', 'estado', 'municipio', 'bioma']

partes_focos = []
for ano in range(ANO_INICIO, ANO_FIM + 1):
    nome = f"focos_br_sp_ref_{ano}"
    caminho = os.path.join(PASTA_FOCOS, nome, f"{nome}.csv")
    df = pd.read_csv(caminho, sep=',', skiprows=1, names=COLUNAS_FOCOS, encoding='utf-8-sig')
    partes_focos.append(df)

focos = pd.concat(partes_focos, ignore_index=True)
print(f"  -> {len(focos)} focos de calor consolidados")

# Estacao mais proxima de cada foco, pela coordenada do proprio
# evento (mais preciso que ir via municipio, ja que o foco tem
# lat/lon exatos).
estacoes_relevantes = estacoes[estacoes['Código'].isin(codigos_estacoes)].reset_index(drop=True)
tree_relevante = BallTree(
    np.radians(estacoes_relevantes[['Latitude', 'Longitude']].astype(float).values), metric='haversine'
)
_, idx_focos = tree_relevante.query(np.radians(focos[['lat', 'lon']].values), k=1)
focos['Codigo_Estacao'] = estacoes_relevantes.iloc[idx_focos.flatten()]['Código'].str.strip().str.upper().values
focos['timestamp'] = pd.to_datetime(focos['data_pas']).dt.floor('h')


# ================================================================
# ETAPA 6 -- Dataframe final: estacao x hora, com contagem de focos
# ================================================================
print("[Etapa 6] Montando o dataframe final (estacao x hora)...")
horas = pd.date_range(f'{ANO_INICIO}-01-01 00:00', f'{ANO_FIM}-12-31 23:00', freq='h')
skeleton = pd.MultiIndex.from_product(
    [codigos_estacoes, horas], names=['Codigo_Estacao', 'timestamp']
).to_frame(index=False)

contagem_focos = focos.groupby(['Codigo_Estacao', 'timestamp']).size().reset_index(name='qtd_focos_calor')

dataframe_final = skeleton.merge(contagem_focos, on=['Codigo_Estacao', 'timestamp'], how='left')
dataframe_final['qtd_focos_calor'] = dataframe_final['qtd_focos_calor'].fillna(0).astype('int16')
dataframe_final = dataframe_final.merge(
    met[['Codigo_Estacao', 'timestamp'] + COLUNAS_NUMERICAS],
    on=['Codigo_Estacao', 'timestamp'], how='left'
)

print(f"  -> {len(dataframe_final)} linhas | {dataframe_final['qtd_focos_calor'].sum()} focos capturados na contagem")

dataframe_final.to_csv(
    os.path.join(PASTA_PROJETO, 'dataframe_final_estacao_hora.csv'), index=False, encoding='utf-8-sig'
)


# ================================================================
# ETAPA 7 -- Preparacao das variaveis para a analise de significancia
# ================================================================
# Reducao por VIF: das 17 variaveis brutas, varias sao quase-
# duplicatas (ex.: 3 medidas de pressao, 4 de temperatura, 3 de
# umidade -- todas com VIF na casa das centenas/milhares na primeira
# checagem). Mantivemos 1 representante por grupo fisico.
print("[Etapa 7] Preparando variaveis (VIF, anomalia por estacao, tendencia de pressao)...")
COLUNAS_REDUZIDAS = [
    'PRECIPITAÇÃO TOTAL, HORÁRIO (mm)',
    'PRESSAO ATMOSFERICA AO NIVEL DA ESTACAO, HORARIA (mB)',
    'RADIACAO GLOBAL (KJ/m²)',
    'TEMPERATURA DO AR - BULBO SECO, HORARIA (°C)',
    'UMIDADE RELATIVA DO AR, HORARIA (%)',
    'VENTO, DIREÇÃO HORARIA (gr) (° (gr))',
    'VENTO, RAJADA MAXIMA (m/s)',
    'VENTO, VELOCIDADE HORARIA (m/s)',
]

# Tendencia de pressao (variacao em 3h) -- calculada sobre o
# esqueleto completo (dataframe_final), nao sobre `met` diretamente,
# porque `met` pode ter horas totalmente ausentes na sequencia (nao
# so NaN), o que faria o .diff(3) comparar horas mais distantes do
# que parece.
dataframe_final = dataframe_final.sort_values(['Codigo_Estacao', 'timestamp']).reset_index(drop=True)
dataframe_final['tendencia_pressao_3h'] = (
    dataframe_final.groupby('Codigo_Estacao')['PRESSAO ATMOSFERICA AO NIVEL DA ESTACAO, HORARIA (mB)'].diff(3)
)

COLUNAS_MODELO = COLUNAS_REDUZIDAS + ['tendencia_pressao_3h']
dados = dataframe_final.dropna(subset=COLUNAS_MODELO).copy()

# Anomalia por estacao (demean): remove o efeito fixo de altitude/
# geografia de cada estacao nas variaveis de NIVEL (pressao,
# temperatura e umidade tem forte correlacao com altitude -- ex.:
# correlacao de -0,998 entre pressao media e altitude entre as
# estacoes). A tendencia de pressao ja e uma diferenca dentro da
# propria estacao, entao nao precisa desse ajuste.
#
# Nota metodologica: a direcao do vento tambem passa por essa
# demeaning antes de virar seno/cosseno abaixo -- foi assim que
# fizemos na analise original. O ideal estatisticamente seria
# decompor o angulo bruto (media circular != media aritmetica), mas
# o efeito pratico na conclusao (direcao do vento e significante)
# deve ser pequeno; vale citar como nota de metodologia se for
# relevante para o rigor do TCC.
medias_por_estacao = dados.groupby('Codigo_Estacao')[COLUNAS_REDUZIDAS].transform('mean')
dados[COLUNAS_REDUZIDAS] = dados[COLUNAS_REDUZIDAS] - medias_por_estacao

# Direcao do vento e uma variavel circular (0 e 360 graus sao a
# mesma direcao) -- decompor em seno/cosseno em vez de usar o grau
# bruto como preditor linear.
graus = np.radians(dados['VENTO, DIREÇÃO HORARIA (gr) (° (gr))'])
dados['vento_dir_sin'] = np.sin(graus)
dados['vento_dir_cos'] = np.cos(graus)

COLUNAS_X = [c for c in COLUNAS_MODELO if c != 'VENTO, DIREÇÃO HORARIA (gr) (° (gr))'] + ['vento_dir_sin', 'vento_dir_cos']
dados['ocorrencia'] = (dados['qtd_focos_calor'] > 0).astype(int)

vif = pd.DataFrame({
    'variavel': COLUNAS_X,
    'VIF': [variance_inflation_factor(dados[COLUNAS_X].values, i) for i in range(len(COLUNAS_X))]
})
print("  -> VIF final (esperado: tudo abaixo de 5):")
print(vif.sort_values('VIF', ascending=False).to_string(index=False))


# ================================================================
# ETAPA 8 -- Regressao logistica (amostragem caso-controle)
# ================================================================
# O evento e raro (~0,27% das horas tem foco de calor). Regressao
# logistica direta em todos os dados gera quase-separacao (parte da
# amostra e "perfeitamente prevista", deixando a estimativa
# instavel). Solucao padrao para evento raro (King & Zeng, 2001):
# manter todos os positivos e amostrar um numero bem maior (aqui,
# 10x) de negativos -- nao enviesa os coeficientes das variaveis, so
# desloca o intercepto (que nao usamos para interpretacao aqui).
print("[Etapa 8] Ajustando a regressao logistica de significancia...")
np.random.seed(42)
positivos = dados[dados['ocorrencia'] == 1]
negativos = dados[dados['ocorrencia'] == 0].sample(n=len(positivos) * 10, random_state=42)
amostra = pd.concat([positivos, negativos])

X = amostra[COLUNAS_X]
X_padronizado = sm.add_constant((X - X.mean()) / X.std())
y = amostra['ocorrencia']

modelo = sm.Logit(y, X_padronizado).fit()
print(modelo.summary())
print("\nRazao de chance por variavel:")
print(np.exp(modelo.params).sort_values(ascending=False))

# Teste de razao de verossimilhanca para a direcao do vento (seno +
# cosseno juntos), ja que nenhuma das duas componentes isoladas tem
# interpretacao individual valida -- o que importa e se a direcao do
# vento, como um todo, melhora o modelo.
modelo_sem_direcao = sm.Logit(y, X_padronizado.drop(columns=['vento_dir_sin', 'vento_dir_cos'])).fit()
lr_stat = 2 * (modelo.llf - modelo_sem_direcao.llf)
p_valor = stats.chi2.sf(lr_stat, df=2)
print(f"\n[Etapa 8] Teste de razao de verossimilhanca (direcao do vento): "
      f"estatistica={lr_stat:.2f}, p={p_valor:.4f}")


# ================================================================
# ETAPA 9 -- Graficos e tabelas resumo para o TCC
# ================================================================
# Gera as figuras e a tabela descritiva usadas para ilustrar a secao
# de Resultados do TCC. Tudo eh calculado a partir dos dataframes ja
# construidos nas etapas anteriores (nao baixa nem recalcula nada).
print("[Etapa 9] Gerando graficos e tabela resumo para o TCC...")
PASTA_FIGURAS = os.path.join(PASTA_PROJETO, 'figuras')
os.makedirs(PASTA_FIGURAS, exist_ok=True)

# Figura 1 -- focos de calor por ano
focos_por_ano = focos.groupby(focos['timestamp'].dt.year).size()
plt.figure(figsize=(8, 5))
focos_por_ano.plot(kind='bar', color='#c0392b')
plt.title('Focos de calor por ano - Estado de Sao Paulo (2015-2024)')
plt.xlabel('Ano')
plt.ylabel('Numero de focos de calor')
plt.xticks(rotation=0)
plt.tight_layout()
plt.savefig(os.path.join(PASTA_FIGURAS, 'focos_por_ano.png'), dpi=150)
plt.close()

# Figura 2 -- 15 estacoes com menor completude media de dados
completude = pd.DataFrame({
    col: met.groupby('Codigo_Estacao')[col].apply(lambda s: s.notna().mean() * 100)
    for col in COLUNAS_NUMERICAS
})
completude['media_geral'] = completude.mean(axis=1)
piores_15 = completude['media_geral'].sort_values().head(15)

plt.figure(figsize=(8, 6))
piores_15.plot(kind='barh', color='#2980b9')
plt.title('15 estacoes com menor completude media de dados')
plt.xlabel('Completude media (%)')
plt.ylabel('Codigo da estacao')
plt.gca().invert_yaxis()
plt.tight_layout()
plt.savefig(os.path.join(PASTA_FIGURAS, 'completude_estacoes.png'), dpi=150)
plt.close()
print(f"  -> completude media da rede (55 estacoes): {completude['media_geral'].mean():.1f}%")

# Figura 3 -- coeficientes padronizados da regressao logistica
coefs = modelo.params.drop('const').sort_values()
cores = ['#c0392b' if modelo.pvalues[v] < 0.05 else '#95a5a6' for v in coefs.index]
plt.figure(figsize=(8, 6))
coefs.plot(kind='barh', color=cores)
plt.axvline(0, color='black', linewidth=0.8)
plt.title('Coeficientes padronizados da regressao logistica')
plt.xlabel('Coeficiente padronizado (vermelho = p < 0,05)')
plt.tight_layout()
plt.savefig(os.path.join(PASTA_FIGURAS, 'coeficientes_regressao.png'), dpi=150)
plt.close()

# Figura 4 -- distribuicao dos focos de calor por bioma
plt.figure(figsize=(7, 7))
focos['bioma'].value_counts().plot(kind='pie', autopct='%1.1f%%')
plt.title('Distribuicao dos focos de calor por bioma')
plt.ylabel('')
plt.tight_layout()
plt.savefig(os.path.join(PASTA_FIGURAS, 'focos_por_bioma.png'), dpi=150)
plt.close()

# Tabela resumo -- estatisticas descritivas das variaveis do modelo
estatisticas = dados[COLUNAS_X].describe().T[['mean', 'std', 'min', 'max']].round(2)
estatisticas.to_csv(os.path.join(PASTA_FIGURAS, 'estatisticas_descritivas.csv'))
print(estatisticas)

print(f"\nGraficos e tabela salvos em: {PASTA_FIGURAS}")
print("\nConcluido.")

[Etapa 1] Lendo catalogo de estacoes do arquivo local...
[Etapa 2] Calculando estacao mais proxima de cada municipio...
  -> 55 estacoes cobrem os 645 municipios de SP
[Etapa 3] Consolidando dados meteorologicos brutos (pode levar alguns minutos)...
